In [ ]:
import os
from google import genai
from google.genai import types

def power_disco_ball(power: bool) -> bool:
    """Powers the spinning disco ball."""
    print(f"Disco ball is {'spinning!' if power else 'stopped.'}")
    return True


def start_music(energetic: bool, loud: bool, bpm: int) -> str:
    """Play some music matching the specified parameters.

    Args:
      energetic: Whether the music is energetic or not.
      loud: Whether the music is loud or not.
      bpm: The beats per minute of the music.

    Returns: The name of the song being played.
    """
    print(f"Starting music! {energetic=} {loud=}, {bpm=}")
    return "Never gonna give you up."


def dim_lights(brightness: float) -> bool:
    """Dim the lights.

    Args:
      brightness: The brightness of the lights, 0.0 is off, 1.0 is full.
    """
    print(f"Lights are now set to {brightness:.0%}")
    return True

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
# Set the model up with tools.
house_fns = [power_disco_ball, start_music, dim_lights]

#關閉自動函式呼叫,才能檢視模型一次回傳的多個 function_call(平行呼叫)
chat = client.chats.create(
    model='gemini-flash-latest',
    config=types.GenerateContentConfig(
        tools=house_fns,
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True)
    )
)
response = chat.send_message("Turn this place into a party!")

# Print out each of the function calls requested from this single call.
for part in response.candidates[0].content.parts:
    if fn := part.function_call:
        args = ", ".join(f"{key}={val}" for key, val in fn.args.items())
        print(f"{fn.name}({args})")

In [ ]:
# Simulate the responses from the specified tools.
responses = {
    "power_disco_ball": True,
    "start_music": "Never gonna give you up.",
    "dim_lights": True,
}

# Build the response parts.
response_parts = [
    types.Part.from_function_response(name=fn, response={"result": val})
    for fn, val in responses.items()
]

response = chat.send_message(response_parts)
print(response.text)